In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH04-Models-Serialization")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH04-Models-Serialization


In [2]:
prompt = PromptTemplate.from_template("{fruit}의 색상이 무엇입니까?")

print(f"ChatOpenAI: {ChatOpenAI.is_lc_serializable()}")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

chain = prompt | llm

chain.is_lc_serializable()


ChatOpenAI: True


True

In [3]:
from langchain_core.load import dumpd, dumps

dumpd_chain = dumpd(chain)

dumpd_chain

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'],
 'kwargs': {'first': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
   'kwargs': {'input_variables': ['fruit'],
    'template': '{fruit}의 색상이 무엇입니까?',
    'template_format': 'f-string'},
   'name': 'PromptTemplate'},
  'last': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
   'kwargs': {'model_name': 'gpt-4o-mini',
    'temperature': 0.0,
    'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']},
    'stream_usage': True},
   'name': 'ChatOpenAI'}},
 'name': 'RunnableSequence'}

In [4]:
type(dumpd_chain)

dict

In [5]:
dumps_chain = dumps(chain)
dumps_chain

'{"lc": 1, "type": "constructor", "id": ["langchain", "schema", "runnable", "RunnableSequence"], "kwargs": {"first": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["fruit"], "template": "{fruit}\\uc758 \\uc0c9\\uc0c1\\uc774 \\ubb34\\uc5c7\\uc785\\ub2c8\\uae4c?", "template_format": "f-string"}, "name": "PromptTemplate"}, "last": {"lc": 1, "type": "constructor", "id": ["langchain", "chat_models", "openai", "ChatOpenAI"], "kwargs": {"model_name": "gpt-4o-mini", "temperature": 0.0, "openai_api_key": {"lc": 1, "type": "secret", "id": ["OPENAI_API_KEY"]}, "stream_usage": true}, "name": "ChatOpenAI"}}, "name": "RunnableSequence"}'

In [6]:
type(dumps_chain)

str

In [7]:
import os
import pickle

os.makedirs("data", exist_ok=True)

with open("data/fruit_chain.pkl", "wb") as f:
    pickle.dump(dumpd_chain, f)

In [8]:
import json

with open("data/fruit_chain.json", "w") as fp:
    json.dump(dumpd_chain, fp)

In [9]:
import pickle

with open("data/fruit_chain.pkl", "rb") as f:
    loaded_chain = pickle.load(f)

In [10]:
from langchain_core.load import load
# ChatOpenAI처럼 langchain_core 바깥(partner 패키지) 클래스까지 복원하려면
# allowed_objects를 명시해야 함. 신뢰할 수 없는 외부 데이터라면 "all" 대신
# 필요한 클래스만 명시적으로 나열하는 것이 더 안전함.
chain_from_file = load(loaded_chain, allowed_objects="all")
print(chain_from_file.invoke({"fruit": "사과"}))

C:\Users\user\AppData\Local\Temp\ipykernel_11828\2152703921.py:5: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  chain_from_file = load(loaded_chain, allowed_objects="all")


content='사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등 여러 가지 색상이 있으며, 품종에 따라 다르게 나타납니다. 예를 들어, 레드 딜리셔스는 주로 빨간색이고, 그라니 스미스는 초록색이며, 골든 딜리셔스는 노란색입니다. 또한, 일부 사과는 두 가지 이상의 색상이 섞여 있을 수도 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 16, 'total_tokens': 117, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f70554c601', 'id': 'chatcmpl-EOwdBRvuRZ5dfNisSwZcXo9PG9hQS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0ad4d-dcb2-7452-89f0-8da98d33b7bf-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 101, 'total_tokens'

In [11]:
from langchain_core.load import load, loads

load_chain = load(
    loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]}, allowed_objects="all"
)

load_chain.invoke({"fruit": "사과"})

AIMessage(content='사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등 여러 가지 색상이 있으며, 품종에 따라 다르게 나타납니다. 예를 들어, 레드 딜리셔스는 주로 빨간색이고, 그라니 스미스는 초록색이며, 골든 딜리셔스는 노란색입니다. 또한, 일부 사과는 두 가지 이상의 색상이 섞여 있는 경우도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 16, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f70554c601', 'id': 'chatcmpl-EOwdCyo84Z1tLORcNUIoBdopoWjNq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ad4d-e406-7ba1-b2ab-a148c00b124c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 10

In [12]:
with open("data/fruit_chain.json", "r") as fp:
    loaded_from_json_chain = json.load(fp)
    loads_chain = load(loaded_from_json_chain, allowed_objects="all")

loads_chain.invoke({"fruit": "사과"})

AIMessage(content='사과의 색상은 다양합니다. 일반적으로 빨간색, 초록색, 노란색 등 여러 가지 색상이 있으며, 품종에 따라 다르게 나타납니다. 예를 들어, 레드 딜리셔스는 주로 빨간색이고, 그라니 스미스는 초록색이며, 골든 딜리셔스는 노란색입니다. 또한, 일부 사과는 두 가지 이상의 색상이 섞여 있는 경우도 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 16, 'total_tokens': 118, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f70554c601', 'id': 'chatcmpl-EOwdDwgHGOfoAMh1h1ZukJjXupDrd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ad4d-e88f-74e3-bfa5-6bc0fdfa6ed1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 10